## 3.4 Linear Regression from Scratch

We now implement linear regression fully from scratch, covering:

- The model
- The loss function
- A minibatch stochastic gradient descent optimizer
- A training function that ties everything together

We then apply this to the synthetic dataset from Section 3.3. While modern frameworks can automate most of this, implementing from scratch ensures a deep understanding — essential when customizing models, layers, or loss functions. We rely only on tensors and automatic differentiation here; a more concise framework-based implementation follows later.

In [ ]:
%matplotlib inline
import torch
from d2l import torch as d2l

### 3.4.1 Define the Model

In [ ]:
class LinearRegressionScratch(d2l.Module):
    """The linear regression model implemented from scratch."""
    def __init__(self, num_inputs, lr, sigma=0.01):
        super().__init__()
        self.save_hyperparameters()  # saves num_inputs, lr, sigma as self attributes
        self.w = torch.nn.Parameter(torch.normal(0, sigma, (num_inputs, 1)))  # weights ~ N(0, sigma)
        self.b = torch.nn.Parameter(torch.zeros(1))                           # bias initialized to 0

@d2l.add_to_class(LinearRegressionScratch)  #@save
def forward(self, X):
    return torch.matmul(X, self.w) + self.b  # y_hat = Xw + b

### 3.4.2 Defining the Loss Function

In [ ]:
@d2l.add_to_class(LinearRegressionScratch) #@save
def loss(self, y_hat, y):
    l = (y_hat - y) ** 2 / 2
    return l.mean()

### 3.4.3 Defining the Optimization Algorithm

In [ ]:
class SGD(d2l.HyperParameters):  #@save
    """Minibatch stochastic gradient descent."""
    def __init__(self, params, lr):
        self.save_hyperparameters()  # saves params and lr as self attributes

    def step(self):
        for param in self.params:
            param -= self.lr * param.grad  # w = w - lr * dL/dw

    def zero_grad(self):
        for param in self.params:
            if param.grad is not None:
                param.grad.zero_()  # reset gradients to 0 before next backward pass

### 3.4.4 Training

In [ ]:
@d2l.add_to_class(d2l.Trainer)  #@save
def prepare_batch(self, batch):
    return batch

@d2l.add_to_class(d2l.Trainer)  #@save
def fit_epoch(self):
    self.model.train()
    for batch in self.train_dataloader:
        loss = self.model.training_step(self.prepare_batch(batch))
        self.optim.zero_grad()       # reset gradients from previous step
        with torch.no_grad():
            loss.backward()          # compute gradients
            if self.gradient_clip_val > 0:  # To be discussed later
                self.clip_gradients(self.gradient_clip_val, self.model)
            self.optim.step()        # update parameters
        self.train_batch_idx += 1

    if self.val_dataloader is None:
        return

    self.model.eval()
    for batch in self.val_dataloader:
        with torch.no_grad():
            self.model.validation_step(self.prepare_batch(batch))
        self.val_batch_idx += 1

In [ ]:
model = LinearRegressionScratch(num_inputs=2, lr=0.03)
data = d2l.SyntheticRegressionData(w=torch.tensor([2, -3.4]), b=4.2)
trainer = d2l.Trainer(max_epochs=3)
trainer.fit(model, data)

In [ ]:
with torch.no_grad():
    print(f'error in estimating w: {data.w - model.w.reshape(data.w.shape)}')
    print(f'error in estimating b: {data.b - model.b}')